# Isolation Forest Training
Train the anomaly detection model on your transaction data. The generated model will be automatically saved to `./models_store` so the API can use it.

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to python path so we can import app modules
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)


In [ ]:
import pandas as pd
from app.features.engineer import engineer_features
from app.models.anomaly import AnomalyDetector
from app.core.config import config


In [ ]:
# Configuration
DATA_PATH = '../test_data.csv'  # Modify this to point to your real CSV
CONTAMINATION = config.ANOMALY_CONTAMINATION
print(f"Using dataset: {DATA_PATH}")


In [ ]:
# 1. Load Dataset
def load_dataset(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    col_map = {}
    for col in df.columns:
        if 'date' in col and 'date' not in col_map.values():
            col_map[col] = 'date'
        elif ('desc' in col or 'narr' in col or 'particular' in col) and 'description' not in col_map.values():
            col_map[col] = 'description'
        elif ('amount' in col or 'debit' in col or 'credit' in col) and 'amount' not in col_map.values():
            col_map[col] = 'amount'
        elif 'bal' in col and 'balance' not in col_map.values():
            col_map[col] = 'balance'
    df = df.rename(columns=col_map)
    if 'date' not in df.columns or 'amount' not in df.columns:
        raise ValueError("CSV must contain 'date' and 'amount'")
    if 'description' not in df.columns:
        df['description'] = 'Unknown'
    if 'balance' not in df.columns:
        df['balance'] = None
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
    df = df.dropna(subset=['date', 'amount'])
    return df[['date', 'description', 'amount', 'balance']].sort_values('date').reset_index(drop=True)

df = load_dataset(DATA_PATH)
print(f"Loaded {len(df)} transactions.")
display(df.head())


In [ ]:
# 2. Engineer Features
featured_df = engineer_features(df)
display(featured_df.head())


In [ ]:
# 3. Train Model & Persist to app/models_store
detector = AnomalyDetector(contamination=CONTAMINATION)
detector.fit(featured_df)
print(f"Model saved to: {detector._model_path}")


In [ ]:
# 4. Evaluate Results
result_df = detector.predict(featured_df)
anomalies = result_df[result_df['is_anomaly']]
print(f"Flagged {len(anomalies)} anomalies out of {len(result_df)} ({len(anomalies)/len(result_df)*100:.1f}%)")
display(anomalies[['date', 'description', 'amount', 'anomaly_score']].head(10))
